# UniDriveVLA — QwenVL3APlanningHead Standalone Export (ONNX-clean)

Exports the **QwenVL3APlanningHead** component as ONNX + TorchScript.

## `if export:` / `else:` pattern
The model code uses an **`export` flag** in 3 files:
- `unidrivevla.py` — `extract_feat()` reshape vs view
- `unified_perception_decoder.py` — `_prepare_bev()`, `forward_stage1()`, `forward_stage2()`
- `qwen_planning_head.py` — `forward_train()`, `forward_test()`, `_feats_to_bev()`

**Workflow:**
```
Same model, same weights
    ┌───────────┴───────────┐
    │                       │
export=False             export=True
(PyTorch forward)        (ONNX export)
    │                       │
Output A                 Output B
    │                       │
    └──── Compare ──────────┘
         (must match)
```

## Full pipeline
```
img  (B, N_cam=6, 3, H=450, W=800)
  → ResNet-50 backbone     C5: (B*N, 2048, 14, 25)
  → FPN neck Conv2d(2048→256)  (B*N, 256, 14, 25)   ← key part
  → BEV camera-avg + pool  (B, 2500, 256)  [50×50 grid]
  → UnifiedPerceptionDecoder Stage 1
  → VLM stub  Linear(256→2048)  [Qwen3-VL-2B placeholder]
  → DirectVLMFusion  Linear(2048→256)
  → UnifiedPerceptionDecoder Stage 2
  → 7 outputs
```

## ONNX-clean fixes (verbose=True confirmed)
| Issue | Root cause | Fix |
|-------|-----------|-----|
| `Unsqueeze` ops | Query params `(N,d)` + `.unsqueeze(0)` | Params stored as `(1,N,d)` |
| `Unsqueeze` | `BEVPositionEncoding` pe `(H*W,d)` + `.unsqueeze(0)` | pe stored as `(1,H*W,d)` |
| `Split(768×256)` | `nn.MultiheadAttention` packed `in_proj_weight` | Replaced with `ExplicitMHA` (separate `q_proj`/`k_proj`/`v_proj`) |

**Result:** Unsqueeze=0, Split=0, Conv=55

## Outputs
| Name | Shape | Description |
|------|-------|-------------|
| `det_cls` | (B, 900, 10) | Detection class logits |
| `det_bbox` | (B, 900, 10) | Box params cx,cy,cz,w,l,h,sin,cos,vx,vy |
| `map_cls` | (B, 100, 3) | Map class logits: divider/ped_crossing/boundary |
| `map_pts` | (B, 100, 20, 2) | Map polyline waypoints |
| `plan_trajs` | (B, 3, 6, 2) | 3 planning modes × 6 steps × (x,y) |
| `plan_scores` | (B, 3) | Planning mode scores |
| `vlm_plan` | (B, 6, 2) | Direct VLM action head: 6 waypoints |

In [ ]:
# ── Cell 1: Check GPU and PyTorch version ─────────────────────────────────────
import subprocess, sys

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('GPU:', result.stdout.strip())
else:
    print('No GPU detected — running on CPU (~3-5 min for export)')

import torch
print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')

In [ ]:
# ── Cell 2: Clone repository ──────────────────────────────────────────────────
import os

REPO_URL  = 'https://github.com/ARNiteshKumar/UniDriveVLA_MulticoreWare.git'
BRANCH    = 'claude/nuscenes-mini-dataset-repo-V4wLx'
REPO_DIR  = '/content/UniDriveVLA_MulticoreWare'

if os.path.exists(REPO_DIR):
    print('Repo already exists — pulling latest...')
    !cd {REPO_DIR} && git pull origin {BRANCH} --rebase
else:
    print(f'Cloning {REPO_URL}  branch={BRANCH} ...')
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Working dir: {os.getcwd()}')
!git log --oneline -3

In [ ]:
# ── Cell 3: Install dependencies ──────────────────────────────────────────────
# torch and torchvision are pre-installed in Colab.
# onnxscript is REQUIRED by PyTorch >= 2.1 dynamo ONNX exporter —
# without it you get:  ModuleNotFoundError: No module named 'onnxscript'
!pip install -q onnx onnxruntime onnxscript

import onnx, onnxruntime, onnxscript
print(f'onnx        : {onnx.__version__}')
print(f'onnxruntime : {onnxruntime.__version__}')
print(f'onnxscript  : {onnxscript.__version__}')

In [ ]:
# ── Cell 4: Run the export ────────────────────────────────────────────────────
#
# Uses the `if export:` / `else:` pattern inside the model code:
#
#   Step 1: Build model with export=False (normal PyTorch path)
#   Step 2: Run PyTorch forward (export=False) → save reference outputs
#   Step 3: Set export=True → ONNX export (ONNX-safe code path)
#   Step 4: Compare PyTorch (export=False) vs ONNX Runtime outputs
#   Step 5: TorchScript export + comparison
#
# Same model, same weights, two code paths → outputs must match.
#
# Files created in exports/:
#   planning_head.onnx        ONNX graph file (~0.9 MB)
#   planning_head.onnx.data   ONNX external weights (~130 MB)
#   planning_head.pt          TorchScript traced model (~130 MB)
#   planning_head_info.json   metadata JSON

!python scripts/export_planning_head.py \
    --output-dir exports/ \
    --img-h 450 \
    --img-w 800 \
    --num-cams 6 \
    --bev-h 50 \
    --bev-w 50

In [ ]:
# ── Cell 5: Show exported files + metadata JSON ───────────────────────────────
import json
from pathlib import Path

print('=== Exported files ===')
for f in sorted(Path('exports').iterdir()):
    print(f'  {f.name:<40}  {f.stat().st_size/1e6:>8.2f} MB')

print()
info = json.loads(Path('exports/planning_head_info.json').read_text())
print('=== planning_head_info.json ===')
print(json.dumps(info, indent=2))

In [ ]:
# ── Cell 6: Inspect ONNX graph — verify Unsqueeze=0, Split=0, Conv=55 ─────────
import onnx
from collections import Counter

model_onnx = onnx.load('exports/planning_head.onnx')
op_counts  = Counter(n.op_type for n in model_onnx.graph.node)

# Input info
inp = model_onnx.graph.input[0]
shape_str = ' x '.join(
    str(d.dim_value) if d.dim_value else d.dim_param
    for d in inp.type.tensor_type.shape.dim
)
print(f'Input  name  : {inp.name}')
print(f'Input  shape : {shape_str}')
print(f'Total nodes  : {len(model_onnx.graph.node)}')
print()

print('Op counts:')
for op, cnt in sorted(op_counts.items(), key=lambda x: -x[1]):
    flag = ''
    if op == 'Conv':      flag = '  <-- backbone + FPN neck'
    if op == 'Unsqueeze': flag = '  <-- SHOULD BE 0 AFTER FIX'
    if op == 'Split':     flag = '  <-- SHOULD BE 0 AFTER FIX'
    print(f'  {op:<22}: {cnt:>4}{flag}')

# Final check
print()
checks = {
    'Unsqueeze = 0': op_counts.get('Unsqueeze', 0) == 0,
    'Split = 0'    : op_counts.get('Split',     0) == 0,
    'Conv > 0'     : op_counts.get('Conv',       0) > 0,
    'Input = img'  : inp.name == 'img',
}
for label, ok in checks.items():
    print(f'  [{"PASS" if ok else "FAIL"}]  {label}')

In [ ]:
# ── Cell 7: ONNX inference — run the exported model ───────────────────────────
import torch
import numpy as np
import onnxruntime as ort

# Camera image input: (batch=1, N_cam=6, RGB, H=450, W=800)
dummy_img = torch.zeros(1, 6, 3, 450, 800)

print('Running ONNX inference (input: img shape', list(dummy_img.shape), ')...')
sess = ort.InferenceSession(
    'exports/planning_head.onnx',
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
)

ort_outs = sess.run(None, {'img': dummy_img.numpy()})

out_names = ['det_cls', 'det_bbox', 'map_cls', 'map_pts',
             'plan_trajs', 'plan_scores', 'vlm_plan']

print('\nONNX output shapes:')
for name, out in zip(out_names, ort_outs):
    print(f'  {name:<14}: {list(out.shape)}')

print('\nPlanning trajectories  plan_trajs (3 modes, 6 steps, xy):')
print(ort_outs[out_names.index('plan_trajs')].round(4))

print('\nVLM direct plan  vlm_plan (6 waypoints):')
print(ort_outs[out_names.index('vlm_plan')].round(4))

In [ ]:
# ── Cell 8: TorchScript inference ─────────────────────────────────────────────
import torch

print('Loading TorchScript model...')
ts_model = torch.jit.load('exports/planning_head.pt', map_location='cpu')
ts_model.eval()

dummy_img = torch.zeros(1, 6, 3, 450, 800)
print('Running TorchScript inference...')
with torch.no_grad():
    ts_outs = ts_model(dummy_img)

out_names = ['det_cls', 'det_bbox', 'map_cls', 'map_pts',
             'plan_trajs', 'plan_scores', 'vlm_plan']

print('\nTorchScript output shapes:')
for name, out in zip(out_names, ts_outs):
    print(f'  {name:<14}: {list(out.shape)}')

In [ ]:
# ── Cell 9: KEY COMPARISON — PyTorch (export=False) vs ONNX Runtime ───────────
#
# This is what the mentor wants: one model, same weights, two code paths.
#   export=False → PyTorch forward (normal training path)
#   export=True  → ONNX export → ONNX Runtime inference
# Outputs must match (max_diff < 1e-4).

import sys, types, torch, numpy as np, onnxruntime as ort

# ── Rebuild the model from the repo (same code as export script) ──
sys.path.insert(0, '.')
exec(open('scripts/export_planning_head.py').read().split('def main')[0])

# 1. Build model with export=False
model = PlanningHeadExportWrapper(bev_h=50, bev_w=50, export=False).eval()
dummy_img = torch.zeros(1, 6, 3, 450, 800)

# 2. PyTorch forward (export=False — normal training path)
print('=== PyTorch forward (export=False) ===')
with torch.no_grad():
    pytorch_outs = model(dummy_img)

out_names = ['det_cls', 'det_bbox', 'map_cls', 'map_pts',
             'plan_trajs', 'plan_scores', 'vlm_plan']

# 3. Load ONNX model (exported with export=True)
print('\n=== ONNX Runtime inference (exported with export=True) ===')
sess = ort.InferenceSession(
    'exports/planning_head.onnx',
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
)
onnx_outs = sess.run(None, {'img': dummy_img.numpy()})

# 4. Compare — same weights, two code paths
print('\n=== Comparison: PyTorch (export=False) vs ONNX Runtime ===')
print('Same model, same weights, two code paths:\n')
all_ok = True
for name, pt_out, ort_out in zip(out_names, pytorch_outs, onnx_outs):
    diff = abs(pt_out.numpy() - ort_out).max()
    ok   = diff < 1e-4
    status = 'PASS' if ok else 'FAIL'
    print(f'  [{status}]  {name:<14}  shape={str(list(pt_out.shape)):<20}  max_diff = {diff:.2e}')
    if not ok:
        all_ok = False

print()
if all_ok:
    print('All outputs match! PyTorch and ONNX Runtime produce identical results.')
    print('The if export: / else: pattern works correctly.')
else:
    print('WARNING: some outputs differ — check the if export: branches.')

In [ ]:
# ── Cell 10: Full summary printout ────────────────────────────────────────────
import json
import onnx
from collections import Counter
from pathlib import Path

info = json.loads(Path('exports/planning_head_info.json').read_text())
g    = onnx.load('exports/planning_head.onnx')
ops  = Counter(n.op_type for n in g.graph.node)

print('=' * 58)
print('  UniDriveVLA — QwenVL3APlanningHead Export Summary')
print('=' * 58)

print('\n  Pipeline:')
for step in info['pipeline']:
    print(f'    {step}')

print(f"\n  Input  : {info['input']['name']}  {info['input']['shape']}")
print(f"           {info['input']['description']}")

print('\n  Intermediate shapes:')
for k, v in info['intermediate_shapes'].items():
    print(f'    {k:<15}: {v}')

print('\n  FPN Neck weight stats:')
for k, v in info['neck_weight_stats'].items():
    print(f'    {k}:')
    for stat, val in v.items():
        print(f'      {stat}: {val}')

print('\n  Outputs:')
for name, meta in info['outputs'].items():
    print(f'    {name:<14}: {meta["shape"]}')

print('\n  Parameters (M):')
for k, v in info['parameters_M'].items():
    print(f'    {k:<15}: {v}')

print('\n  File sizes:')
for k, v in info['file_sizes_MB'].items():
    print(f'    {k:<15}: {v} MB')

print('\n  ONNX graph op check:')
print(f'    Total nodes : {len(g.graph.node)}')
print(f'    Unsqueeze   : {ops.get("Unsqueeze", 0)}  (target: 0)')
print(f'    Split       : {ops.get("Split",     0)}  (target: 0)')
print(f'    Conv        : {ops.get("Conv",       0)}  (backbone+neck present)')
print(f'    MatMul      : {ops.get("MatMul",     0)}  (ExplicitMHA Q/K/V proj)')
print('=' * 58)

In [ ]:
# ── Cell 11: Download exported files ──────────────────────────────────────────
from google.colab import files
import os

to_download = [
    ('exports/planning_head.onnx',      'ONNX graph file  (needs .data file alongside it)'),
    ('exports/planning_head.onnx.data', 'ONNX external weights (~130 MB)'),
    ('exports/planning_head.pt',        'TorchScript traced model (~130 MB)'),
    ('exports/planning_head_info.json', 'Metadata: shapes, params, neck stats'),
]

for fp, desc in to_download:
    if os.path.exists(fp):
        sz = os.path.getsize(fp) / 1e6
        print(f'Downloading  {fp}  ({sz:.1f} MB)  --  {desc}')
        files.download(fp)
    else:
        print(f'Not found: {fp}')